# Official StyleGAN2-ADA: fresh CUDA runtime verification

This prepared notebook has not itself been executed. Its script consolidates the operations and API patches tested successfully in notebook 11 on 11 November 2026 (PyTorch 2.11.0+cu130, CUDA 13.0, Tesla T4). It clones the pinned NVIDIA source, applies recorded loader/grid API compatibility patches, and runs synthetic CUDA/gradient checks. No real-image dataset, pretrained weights, optimizer, or GAN fitting is used. Full training and path-length regularization are not validated by these checks. Use a new output folder for every attempt.


In [ ]:
from pathlib import Path
script_path = Path("/kaggle/working/verify_stylegan2ada_runtime.py")
script_path.write_text('"""Verify pinned StyleGAN2-ADA on CUDA without training or reading image data.\n\nTwo recorded compatibility patches: extension module loading and NVIDIA\'s\nnewer grid-gradient API. No network architecture or CUDA kernel changes.\nRequires a fresh output folder and a T4-compatible PyTorch/CUDA runtime.\n"""\nimport argparse\nfrom pathlib import Path\nparser = argparse.ArgumentParser(description=__doc__)\nparser.add_argument(\'--output\', type=Path, required=True)\nargs = parser.parse_args()\noutput_root = args.output.resolve()\nassert not output_root.exists(), \'Use a fresh output directory\'\noutput_root.mkdir(parents=True)\n\n\nfrom pathlib import Path\nimport subprocess, sys, json, platform, importlib.util\n\nUPSTREAM_COMMIT = \'d72cc7d041b42ec8e806021a205ed9349f87c6a4\'\nwork = output_root/\'source_checkout\'\nassert not work.exists(), \'Choose a fresh work folder; preserve previous evidence\'\nwork.mkdir()\nupstream = work / \'stylegan2-ada-pytorch\'\nsubprocess.run([\'git\',\'clone\',\'https://github.com/NVlabs/stylegan2-ada-pytorch.git\',str(upstream)], check=True)\nsubprocess.run([\'git\',\'checkout\',\'--detach\',UPSTREAM_COMMIT], cwd=upstream, check=True)\nactual = subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],cwd=upstream,text=True).strip()\nassert actual == UPSTREAM_COMMIT\nrequired = [\'torch\',\'click\',\'requests\',\'scipy\',\'psutil\',\'tqdm\',\'PIL\',\'ninja\',\'imageio\',\'imageio_ffmpeg\']\nmissing = [name for name in required if importlib.util.find_spec(name) is None]\nassert not missing, f\'Install official dependencies in a compatible runtime first: {missing}\'\nimport torch\nenvironment = dict(python=platform.python_version(), torch=torch.__version__,\n                   cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),\n                   upstream_commit=actual)\n(work/\'environment.json\').write_text(json.dumps(environment,indent=2))\nprint(environment)\n\n\nfrom pathlib import Path\nimport platform, shutil, subprocess, json, torch\nRUN_FEASIBILITY = False\ncheck = dict(python=platform.python_version(), torch=torch.__version__, torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(), gpu_count=torch.cuda.device_count(), nvidia_smi=shutil.which(\'nvidia-smi\'), nvcc=shutil.which(\'nvcc\'), training_launched=False)\nfor name in [\'nvidia-smi\',\'nvcc\']:\n    executable = shutil.which(name)\n    if executable:\n        args = [executable] + ([\'--version\'] if name == \'nvcc\' else [\'--query-gpu=name,driver_version,memory.total\',\'--format=csv,noheader\'])\n        r = subprocess.run(args,capture_output=True,text=True)\n        check[name+\'_output\'] = r.stdout or r.stderr\nif torch.cuda.is_available():\n    x = torch.arange(16,device=\'cuda\',dtype=torch.float32)\n    check[\'cuda_tensor_test\'] = float((x*x).sum().cpu()) == 1240.0\n    check[\'gpu_names\'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n(output_root/\'gpu_runtime_check.json\').write_text(json.dumps(check,indent=2))\nprint(json.dumps(check,indent=2))\nprint(\'GPU_RUNTIME_CHECK_COMPLETE\')\n\nassert check[\'cuda_available\'] and check.get(\'cuda_tensor_test\'), \'CUDA runtime unavailable\'\n\n\nimport hashlib, difflib, importlib\nloader = upstream/\'torch_utils/custom_ops.py\'\noriginal_loader = loader.read_text()\nassert hashlib.sha256(original_loader.encode()).hexdigest() == \'ee123ae0458fbe463000d5962a5b804d9e91b25d3fda8500eeaa6ddf6b5c932a\'\nassert original_loader.count(\'torch.utils.cpp_extension.load(\') == 2\nassert original_loader.count(\'module = importlib.import_module(module_name)\') == 1\npatched_loader = original_loader.replace(\'torch.utils.cpp_extension.load(\', \'module = torch.utils.cpp_extension.load(\').replace(\'        module = importlib.import_module(module_name)\\n\',\'\')\nloader.write_text(patched_loader)\npatch_text = \'\'.join(difflib.unified_diff(original_loader.splitlines(True),patched_loader.splitlines(True),fromfile=\'a/torch_utils/custom_ops.py\',tofile=\'b/torch_utils/custom_ops.py\'))\n(output_root/\'stylegan_pytorch_loader.patch\').write_text(patch_text)\npatch_info = dict(upstream_commit=UPSTREAM_COMMIT, file=\'torch_utils/custom_ops.py\', original_sha256=hashlib.sha256(original_loader.encode()).hexdigest(), patched_sha256=hashlib.sha256(patched_loader.encode()).hexdigest(), reason=\'Use cpp_extension.load returned Python module instead of a second import by name.\', architecture_changed=False, cuda_kernel_source_changed=False)\n(output_root/\'stylegan_loader_patch.json\').write_text(json.dumps(patch_info,indent=2))\nprint(\'LOADER_COMPATIBILITY_PATCH_APPLIED\')\n\n\ngrid_file = upstream/\'torch_utils/ops/grid_sample_gradfix.py\'\ngrid_original = grid_file.read_text()\nassert hashlib.sha256(grid_original.encode()).hexdigest() == \'a49a8195b734ecba95c9b63d035f26faa5d794a0b26eb3e9d5f3c521be29c536\'\nstart = grid_original.index(\'def _should_use_custom_op():\')\nend = grid_original.index(\'#----------------------------------------------------------------------------\',start)\ngrid_patched = grid_original[:start] + \'def _should_use_custom_op():\\n    return enabled\\n\\n\' + grid_original[end:]\nold_call = "        op = torch._C._jit_get_operation(\'aten::grid_sampler_2d_backward\')\\n        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)"\nnew_call = "        op = torch._C._jit_get_operation(\'aten::grid_sampler_2d_backward\')[0]\\n        output_mask = (ctx.needs_input_grad[1], ctx.needs_input_grad[2])\\n        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, output_mask)"\nassert grid_patched.count(old_call) == 1\ngrid_patched = grid_patched.replace(old_call,new_call)\ngrid_file.write_text(grid_patched)\ngrid_patch = \'\'.join(difflib.unified_diff(grid_original.splitlines(True),grid_patched.splitlines(True),fromfile=\'a/torch_utils/ops/grid_sample_gradfix.py\',tofile=\'b/torch_utils/ops/grid_sample_gradfix.py\'))\n(output_root/\'stylegan_grid_gradfix.patch\').write_text(grid_patch)\ngrid_patch_info = dict(source=\'https://github.com/NVlabs/stylegan3/blob/main/torch_utils/ops/grid_sample_gradfix.py\', changes=\'Port NVIDIA newer API handling: use enabled gradfix, operation tuple[0], backward output_mask. Target runtime PyTorch 2.11 only.\', original_sha256=hashlib.sha256(grid_original.encode()).hexdigest(), patched_sha256=hashlib.sha256(grid_patched.encode()).hexdigest(), architecture_changed=False)\n(output_root/\'stylegan_grid_patch.json\').write_text(json.dumps(grid_patch_info,indent=2))\nprint(\'NVIDIA_GRID_API_COMPATIBILITY_UPDATE_APPLIED\')\n\n\nimport os, sys, traceback, warnings\nos.environ[\'MAX_JOBS\'] = \'2\'\nos.environ[\'TORCH_CUDA_ARCH_LIST\'] = \'.\'.join(map(str,torch.cuda.get_device_capability(0)))\nos.environ[\'TORCH_EXTENSIONS_DIR\'] = str(output_root/\'cuda_extensions\')\nsys.path.insert(0,str(upstream))\nfrom torch_utils.ops import bias_act, upfirdn2d\nfrom torch_utils import custom_ops\ncustom_ops.verbosity = \'full\'\nops_report = dict(environment=environment, upstream_commit=UPSTREAM_COMMIT, training_launched=False, optimizer_updates=0, tests={})\nfor name, module in [(\'bias_act\',bias_act),(\'upfirdn2d\',upfirdn2d)]:\n    print(\'COMPILING\',name,flush=True)\n    try:\n        assert module._init(), \'Compiled plugin unavailable; fallback is not a compilation pass\'\n        x = torch.randn(2,8,16,16,device=\'cuda\',requires_grad=True)\n        if name == \'bias_act\':\n            b = torch.randn(8,device=\'cuda\')\n            y = module.bias_act(x,b,act=\'swish\',impl=\'cuda\')\n            yr = module.bias_act(x,b,act=\'swish\',impl=\'ref\')\n        else:\n            f = module.setup_filter([1,3,3,1],device=\'cuda\')\n            y = module.upfirdn2d(x,f,padding=2,impl=\'cuda\')\n            yr = module.upfirdn2d(x,f,padding=2,impl=\'ref\')\n        torch.testing.assert_close(y,yr,rtol=1e-4,atol=1e-5)\n        g = torch.autograd.grad(y.square().mean(),x,create_graph=True)[0]\n        gr = torch.autograd.grad(yr.square().mean(),x,create_graph=True)[0]\n        torch.testing.assert_close(g,gr,rtol=1e-4,atol=1e-5)\n        gg = torch.autograd.grad(g.square().sum(),x)[0]\n        ggr = torch.autograd.grad(gr.square().sum(),x)[0]\n        torch.testing.assert_close(gg,ggr,rtol=2e-4,atol=1e-5)\n        assert torch.isfinite(gg).all()\n        torch.cuda.synchronize()\n        ops_report[\'tests\'][name] = dict(passed=True, compiled_plugin_loaded=module._plugin is not None, max_forward_error=float((y-yr).abs().max().detach().cpu()), forward_backward_second_derivative=True)\n    except Exception:\n        ops_report[\'tests\'][name] = dict(passed=False,error=traceback.format_exc())\n    (output_root/\'stylegan_cuda_ops_check.json\').write_text(json.dumps(ops_report,indent=2))\n    print(name,ops_report[\'tests\'][name],flush=True)\nops_report[\'all_passed\'] = all(v[\'passed\'] for v in ops_report[\'tests\'].values())\n(output_root/\'stylegan_cuda_ops_check.json\').write_text(json.dumps(ops_report,indent=2))\nprint(\'OFFICIAL_CUDA_OPS_CHECK_COMPLETE\',ops_report[\'all_passed\'])\n\n\nwarnings.filterwarnings(\'once\',message=\'conv2d_gradfix not supported.*\')\nfrom training.networks import Generator, Discriminator\nfrom training.augment import AugmentPipe\nfrom torch_utils.ops import conv2d_gradfix, grid_sample_gradfix\nconv2d_gradfix.enabled = True\ngrid_sample_gradfix.enabled = True\nnetwork_check = dict(training_launched=False, optimizer_updates=0, real_images_read=False, resolution=128, c_dim=2, tests={})\ntry:\n    torch.manual_seed(1142)\n    G = Generator(z_dim=512,c_dim=2,w_dim=512,img_resolution=128,img_channels=3,mapping_kwargs=dict(num_layers=2),synthesis_kwargs=dict(channel_base=16384,channel_max=512)).cuda()\n    D = Discriminator(c_dim=2,img_resolution=128,img_channels=3,channel_base=16384,channel_max=512).cuda()\n    c = torch.eye(2,device=\'cuda\')\n    z = torch.randn(2,512,device=\'cuda\')\n    fake = G(z,c,noise_mode=\'const\')\n    score = D(fake,c)\n    score.mean().backward()\n    assert torch.isfinite(fake).all() and all(torch.isfinite(p.grad).all() for p in G.parameters() if p.grad is not None)\n    network_check[\'tests\'][\'generator_discriminator_backward\'] = True\n    G.zero_grad(set_to_none=True)\n    D.zero_grad(set_to_none=True)\n    a = AugmentPipe(xflip=1,rotate90=1,xint=1,scale=1,rotate=1,aniso=1,xfrac=1).cuda()\n    a.p.copy_(torch.ones_like(a.p))\n    x = torch.randn(2,3,128,128,device=\'cuda\',requires_grad=True)\n    logits = D(a(x),c)\n    with conv2d_gradfix.no_weight_gradients():\n        dx = torch.autograd.grad(logits.sum(),x,create_graph=True)[0]\n    dx.square().sum((1,2,3)).mean().backward()\n    assert torch.isfinite(dx).all() and all(torch.isfinite(p.grad).all() for p in D.parameters() if p.grad is not None)\n    network_check[\'tests\'][\'augmentation_and_r1_double_backward\'] = True\n    network_check[\'passed\'] = True\nexcept Exception:\n    network_check[\'passed\'] = False\n    network_check[\'error\'] = traceback.format_exc()\nfinally:\n    (output_root/\'stylegan_network_check.json\').write_text(json.dumps(network_check,indent=2))\n    print(json.dumps(network_check,indent=2))\n    print(\'SYNTHETIC_NETWORK_CHECK_COMPLETE\')\n\n\nruntime_report = dict(status=\'gpu_and_targeted_runtime_checks_passed\' if ops_report[\'all_passed\'] and network_check[\'passed\'] else \'runtime_check_failed\', hardware=check, cuda_ops=ops_report, network_check=network_check, loader_patch=patch_info, grid_patch=grid_patch_info, conv2d_gradfix_native_fallback=True, training_launched=False, optimizer_updates=0, real_image_content_read=False, limitations=\'Targeted synthetic checks only; full training, path-length regularization, checkpointing and model quality remain unvalidated.\')\n(output_root/\'stylegan_gpu_runtime_report.json\').write_text(json.dumps(runtime_report,indent=2))\nprint(json.dumps(runtime_report,indent=2))\nassert ops_report[\'all_passed\'] and network_check[\'passed\'], \'Runtime validation failed; do not train\'\n')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,str(script_path),"--output","/kaggle/working/stylegan_runtime_verification"],check=True)
